# Log Scales: Semi-log and Log-log Plots

This notebook turns the logarithm into a **lens for data**. Put a logarithmic scale on one axis and every exponential becomes a straight line; put it on both axes and every power law becomes a straight line. The slope of that line is the growth rate or the exponent — read off with a ruler, or fitted with `np.polyfit`. We use the lens to rediscover Kepler's third law from the planets' orbits.

This is notebook 03 of module 02 (**Graphs, equations and scales**) in the **Learning Logarithms** series.

## 1. Introduction

Here are the eight planets, with their mean distance from the Sun $a$ (in astronomical units, Earth $= 1$) and orbital period $T$ (in years):

| | Mercury | Venus | Earth | Mars | Jupiter | Saturn | Uranus | Neptune |
|---|---|---|---|---|---|---|---|---|
| $a$ (AU) | 0.387 | 0.723 | 1.000 | 1.524 | 5.203 | 9.537 | 19.19 | 30.07 |
| $T$ (years) | 0.241 | 0.615 | 1.000 | 1.881 | 11.86 | 29.46 | 84.01 | 164.8 |

Is there a law connecting $a$ and $T$? Plot $T$ against $a$ on ordinary axes and the four inner planets — where all of human spaceflight happens — are squashed into a corner a few millimetres wide, while Neptune stretches the axes. You see a curve bending upwards, but which curve? $a^2$? $a^{1.5}$? $e^{a}$? By eye, on linear axes, you cannot tell.

The same problem appears with anything that spans **many orders of magnitude**: transistor counts on chips (from thousands to tens of billions), earthquake energies, city sizes, animal body masses, the frequencies of words. A linear axis spends all its space on the largest values. A logarithmic axis gives each **factor of ten** the same amount of room — the core picture of this series, applied to a chart.

On a log-log plot the eight planets fall on a perfectly straight line of slope $1.5$. That is Kepler's third law, $T^2 = a^3$ (in these units), found in 1619 from tables of numbers — and now visible in one picture.

**Prerequisites**

- Graphs of $b^x$ and $\log_b x$ — [notebook 02/01](./01_graphs_of_exp_and_log.ipynb).
- Solving exponential equations by taking logs — [notebook 02/02](./02_solving_exponential_and_log_equations.ipynb).
- The laws of logarithms, especially $\log_b(x^k) = k\log_b x$ — [notebook 00/03](../00_foundations/03_laws_of_logarithms.ipynb).
- Straight lines $y = mx + c$ and fitting a line to points by least squares (school algebra; `np.polyfit` does the fitting).

After completing this notebook you will be able to:

- Explain why a log axis gives equal space to equal ratios and how to read values off it.
- Recognise an exponential as a straight line on a semi-log plot, and a power law as a straight line on a log-log plot.
- Read a growth factor, doubling time or exponent from the slope of such a line.
- Fit exponentials and power laws with `np.polyfit` on logged data, and say why that is reasonable for multiplicative noise.
- Diagnose which model fits a data set by checking which kind of plot straightens it.

## 2. Theory

### 2.1 What a log axis does

On a logarithmic axis the value $y$ is drawn at position $\log_{10} y$. Consequences:

- $1, 10, 100, 1000$ are **equally spaced** — each decade gets the same length.
- $1, 2, 4, 8$ are also equally spaced (each step is $\log_{10} 2 \approx 0.301$ of a decade).
- Zero and negative values cannot appear: $\log_{10} 0$ is $-\infty$. The axis goes on forever downward through $0.1, 0.01, \dots$
- Within a decade the ticks $2, 3, \dots, 9$ crowd towards the top: $2$ sits $30\%$ of the way up, $5$ sits $70\%$ of the way.

**Semi-log** plot: one axis logarithmic (usually $y$), the other linear. **Log-log** plot: both logarithmic.

### 2.2 Exponentials are straight on semi-log axes

Take logs of an exponential $y = A \cdot b^{x}$:

$$
\underbrace{y = A\, b^{x}}_{\text{exponent form}} \quad\Longleftrightarrow\quad \underbrace{\log_{10} y = \log_{10} A + (\log_{10} b)\, x}_{\text{a straight line in } (x,\ \log_{10} y)}.
$$

The vertical position is linear in $x$. Slope $m = \log_{10} b$ is measured in **decades per unit of $x$**, and translates into

$$
\text{growth factor per unit } b = 10^{m}, \qquad \text{doubling time } = \frac{\log_{10} 2}{m}, \qquad \text{continuous rate } r = m \ln 10.
$$

(If you fit with $\ln$ instead of $\log_{10}$, the slope is $r = \ln b$ directly.)

### 2.3 Power laws are straight on log-log axes

Take logs of a power law $y = C \cdot x^{k}$:

$$
\underbrace{y = C\, x^{k}}_{\text{exponent form}} \quad\Longleftrightarrow\quad \underbrace{\log_{10} y = \log_{10} C + k\, \log_{10} x}_{\text{a straight line in } (\log_{10} x,\ \log_{10} y)}.
$$

Now the slope is the **exponent** $k$ itself, and it does not depend on which base of logarithm you use (both sides are scaled by the same factor). Reading it off a plot:

$$
k = \frac{\text{decades up}}{\text{decades across}} = \frac{\log_{10}(y_2 / y_1)}{\log_{10}(x_2 / x_1)}.
$$

The intercept, where the line crosses $x = 1$ ($\log_{10} x = 0$), is $C$.

### 2.4 The diagnostic

| Model | Linear axes | Semi-log ($y$ log) | Semi-log ($x$ log) | Log-log |
|---|---|---|---|---|
| $y = A b^x$ (exponential) | curves up | **straight**, slope $\log_{10} b$ | curves up | curves up |
| $y = C x^k$ (power law) | curves (up if $k > 1$) | curves down | curves up | **straight**, slope $k$ |
| $y = A + B\ln x$ (logarithmic) | curves down | curves down | **straight**, slope $B\ln 10$ | curves |

So: plot the data several ways and see which one straightens it. On log-log axes an exponential bends upward ever more steeply, because $\log_{10} y$ grows like $x = 10^{\log_{10} x}$; on semi-log axes a power law bends over, because $\log_{10} y$ grows like $k\log_{10} x$. (Positive $k$ assumed in the table.)

### 2.5 Fitting a line to logged data

To fit $y = C x^k$ to data $(x_i, y_i)$, fit a straight line to $(\log_{10} x_i, \log_{10} y_i)$: `np.polyfit(np.log10(x), np.log10(y), 1)` returns $[k, \log_{10} C]$. For an exponential, fit $(x_i, \log_{10} y_i)$ instead.

This minimises errors in $\log_{10} y$, that is, **relative** errors: a point $10\%$ too high counts the same whether $y$ is $10$ or $10^{10}$. That is the right choice when the scatter is **multiplicative** (measurements are off by a percentage), which is typical for data spanning many decades. Fitting the raw values instead would let the few largest points dominate completely.

Two cautions: the data must be positive, and a straight line on a log-log plot over one or two decades is weak evidence of a power law — many curves look straight over a short range.

## 3. Worked Examples

### Example 1 — reading an exponent from two points

A line on a log-log plot passes through $(10, 3)$ and $(1000, 300{,}000)$. Across: $\log_{10}(1000/10) = 2$ decades. Up: $\log_{10}(300000/3) = 5$ decades. So $k = 5/2 = 2.5$. From $3 = C\cdot 10^{2.5}$: $C = 3 / 316.2 \approx 0.009487$. The law is $y \approx 0.0095\, x^{2.5}$.

### Example 2 — reading a doubling time from a semi-log line

A semi-log plot shows a straight line from $10^4$ in the year 2000 to $10^6$ in 2010. Slope: $2$ decades in $10$ years, $m = 0.2$ decades per year. Growth factor per year: $10^{0.2} \approx 1.585$ ($58.5\%$ per year). Doubling time: $\log_{10} 2 / 0.2 = 0.30103 / 0.2 \approx 1.51$ years.

### Example 3 — Kepler's third law as a line

If $T = a^{1.5}$ then $\log_{10} T = 1.5\,\log_{10} a$: slope $1.5$, through the origin (Earth: $a = 1$, $T = 1$). Check with Jupiter: $5.203^{1.5} = 5.203 \times \sqrt{5.203} \approx 5.203 \times 2.281 \approx 11.87$ years; the table says $11.86$ ✓.

### Example 4 — the base does not matter on log-log axes

For $y = 4x^3$: $\log_{10} y = \log_{10} 4 + 3\log_{10} x$ and $\ln y = \ln 4 + 3\ln x$. Both slopes are $3$. On a semi-log plot of $y = 4 \cdot 2^x$, though, the slope is $\log_{10} 2 \approx 0.301$ with $\log_{10}$ and $\ln 2 \approx 0.693$ with $\ln$ — the same line measured in different units (decades versus factors of $e$).

### Example 5 — where is $5$ on a log axis?

On a decade of length $10$ cm, $5$ sits at $10 \cdot \log_{10} 5 \approx 6.99$ cm from the $1$, and $2$ at $3.01$ cm. Together they make $10$ cm — because $2 \times 5 = 10$. Adding lengths multiplies values: this is the slide rule (notebook 03/02).

## 4. Python Examples

### 4.1 Positions on a log axis

Where each value sits within a decade, as a fraction of the decade's length.

In [ ]:
import math

for value in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    fraction = math.log10(value)
    bar = "#" * round(40 * fraction)
    print(f"{value:>2}: {fraction:.3f} of a decade  |{bar}")

### 4.2 Kepler's third law with `np.polyfit`

Fit a line to $(\log_{10} a, \log_{10} T)$. The slope is the exponent, $10^{\text{intercept}}$ the constant.

In [ ]:
import numpy as np

planets = ["Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune"]
semi_major_axis_au = np.array([0.387, 0.723, 1.000, 1.524, 5.203, 9.537, 19.19, 30.07])
period_years = np.array([0.2408, 0.6152, 1.000, 1.8808, 11.862, 29.457, 84.01, 164.8])

slope, intercept = np.polyfit(np.log10(semi_major_axis_au), np.log10(period_years), 1)
print(f"fitted exponent k = {slope:.4f}")
print(f"fitted constant C = 10^{intercept:.5f} = {10**intercept:.4f}")
print()
print("T^2 / a^3 for each planet (Kepler: should be 1):")
for name, a, T in zip(planets, semi_major_axis_au, period_years):
    print(f"  {name:8s} {T**2 / a**3:.4f}")

### 4.3 A synthetic Moore's-law data set

We **generate** data for illustration (these are not real chip counts): start at $2300$ transistors in 1971 — the size of the first commercial microprocessor — double every two years, and multiply each point by random noise of typical size $\pm 30\%$. Then fit $\log_{10} N$ against the year and recover the doubling time.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=1971)
years = np.arange(1971, 2022, 2)
true_doubling_time = 2.0
noise_factor = np.exp(rng.normal(0.0, 0.3, size=years.size))   # multiplicative noise
transistors = 2300 * 2 ** ((years - 1971) / true_doubling_time) * noise_factor

slope, intercept = np.polyfit(years, np.log10(transistors), 1)
fitted_doubling_time = np.log10(2) / slope
print(f"{years.size} points, from {transistors[0]:.3g} to {transistors[-1]:.3g} transistors")
print(f"slope: {slope:.4f} decades per year -> growth factor {10**slope:.3f} per year")
print(f"fitted doubling time: {fitted_doubling_time:.2f} years (true value {true_doubling_time})")

### 4.4 Fitting raw values versus logged values

Why not fit the exponential to the raw counts $N$ directly? An ordinary least-squares fit adds up squared errors in $N$, and errors scale with $N$ — so each point's influence is roughly proportional to $N^2$. The cell measures that share: the last few points carry essentially all of it, and the first twenty (four decades of history) almost none. The log fit treats every point's *percentage* error equally.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=1971)
years = np.arange(1971, 2022, 2)
transistors = 2300 * 2 ** ((years - 1971) / 2.0) * np.exp(rng.normal(0.0, 0.3, size=years.size))

# log fit (straight line in log10 N)
slope, intercept = np.polyfit(years, np.log10(transistors), 1)
log_fit = 10 ** (intercept + slope * years)

# weight of each point in an unweighted least-squares fit of the raw values ~ its size squared
share_of_sum_of_squares = transistors**2 / np.sum(transistors**2)
print(f"share of sum of squares carried by the last 3 points: {share_of_sum_of_squares[-3:].sum():.1%}")
print(f"share carried by the first 20 points             : {share_of_sum_of_squares[:20].sum():.2e}")
print()
relative_errors = np.abs(log_fit / transistors - 1)
print(f"log fit: median relative error {np.median(relative_errors):.1%}, worst {relative_errors.max():.1%}")

### 4.5 Exponent from two points, as a function

In [ ]:
import math


def power_law_through(point_1, point_2):
    (x1, y1), (x2, y2) = point_1, point_2
    exponent = math.log10(y2 / y1) / math.log10(x2 / x1)
    constant = y1 / x1**exponent
    return exponent, constant


k, C = power_law_through((10, 3), (1000, 300_000))
print(f"Example 1: k = {k:.3f}, C = {C:.6f}")
k, C = power_law_through((1.000, 1.000), (30.07, 164.8))
print(f"Earth -> Neptune: k = {k:.4f}, C = {C:.3f}")

## 5. Visualizations

### 5.1 The planets on linear and log-log axes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

planets = ["Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune"]
a = np.array([0.387, 0.723, 1.000, 1.524, 5.203, 9.537, 19.19, 30.07])
T = np.array([0.2408, 0.6152, 1.000, 1.8808, 11.862, 29.457, 84.01, 164.8])
slope, intercept = np.polyfit(np.log10(a), np.log10(T), 1)
a_line = np.logspace(np.log10(0.25), np.log10(50), 100)

fig, (ax_linear, ax_loglog) = plt.subplots(1, 2, figsize=(12, 5))
ax_linear.plot(a, T, "o", color="tab:blue")
for name, x, y in zip(planets[4:], a[4:], T[4:]):
    ax_linear.annotate(name, (x, y), textcoords="offset points", xytext=(6, -12))
ax_linear.annotate("Mercury, Venus,\nEarth, Mars", (1, 1), textcoords="offset points", xytext=(15, 35),
                   arrowprops=dict(arrowstyle="->"))
ax_linear.set_xlabel("distance a (AU, linear scale)")
ax_linear.set_ylabel("period T (years, linear scale)")
ax_linear.set_title("Linear axes: the inner planets vanish")
ax_linear.grid(True, alpha=0.3)

ax_loglog.plot(a, T, "o", color="tab:blue")
ax_loglog.plot(a_line, 10**intercept * a_line**slope, "--", color="tab:red", label=f"fit: T = a^{slope:.3f}")
for name, x, y in zip(planets, a, T):
    ax_loglog.annotate(name, (x, y), textcoords="offset points", xytext=(6, -12), fontsize=9)
ax_loglog.set_xscale("log")
ax_loglog.set_yscale("log")
ax_loglog.set_xlabel("distance a (AU, log scale)")
ax_loglog.set_ylabel("period T (years, log scale)")
ax_loglog.set_title("Log-log axes: a straight line of slope 1.5")
ax_loglog.grid(True, which="both", alpha=0.3)
ax_loglog.legend(loc="upper left")
plt.tight_layout()
plt.show()

On the right, every factor of $10$ in distance is the same width, and every factor of $10$ in period the same height. Going from Earth to Saturn is one decade across ($\approx 9.5\times$) and one and a half decades up ($\approx 29.5\times$): slope $1.5$.

### 5.2 The diagnostic: one exponential and one power law, three ways

$y = 2^x$ (exponential) and $y = x^3$ (power law) drawn on linear, semi-log and log-log axes. Each is straight on exactly one kind of plot.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(1, 20, 300)
exponential = 2.0**x
power_law = x**3

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
settings = [("linear", "linear", "linear axes"),
            ("linear", "log", "semi-log (y log): exponential straight"),
            ("log", "log", "log-log: power law straight")]
for ax, (x_scale, y_scale, title) in zip(axes, settings):
    ax.plot(x, exponential, color="tab:green", label=r"$2^x$")
    ax.plot(x, power_law, color="tab:purple", label=r"$x^3$")
    ax.set_xscale(x_scale)
    ax.set_yscale(y_scale)
    ax.set_xlabel(f"x ({x_scale} scale)")
    ax.set_ylabel(f"y ({y_scale} scale)")
    ax.set_title(title, fontsize=10)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

The two curves cross twice — near $x \approx 1.37$ and at $x \approx 9.94$ — and after the second crossing the exponential wins forever. On the log-log panel the exponential bends ever more steeply upwards; on the semi-log panel the power law bends over. Notebook 04 is about exactly this race.

### 5.3 Moore's law (synthetic) on linear and semi-log axes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(seed=1971)
years = np.arange(1971, 2022, 2)
transistors = 2300 * 2 ** ((years - 1971) / 2.0) * np.exp(rng.normal(0.0, 0.3, size=years.size))
slope, intercept = np.polyfit(years, np.log10(transistors), 1)
fit_line = 10 ** (intercept + slope * years)

fig, (ax_linear, ax_semilog) = plt.subplots(1, 2, figsize=(12, 4.6))
for ax in (ax_linear, ax_semilog):
    ax.plot(years, transistors, "o", color="tab:blue", label="synthetic data")
    ax.plot(years, fit_line, "-", color="tab:red", label=f"fit: doubling every {np.log10(2) / slope:.2f} years")
    ax.set_xlabel("year (linear scale)")
    ax.legend(loc="upper left")
ax_linear.set_ylabel("transistors (linear scale)")
ax_linear.set_title("Linear: 40 years of data look like zero")
ax_linear.grid(True, alpha=0.3)
ax_semilog.set_yscale("log")
ax_semilog.set_ylabel("transistors (log scale)")
ax_semilog.set_title("Semi-log: a straight line")
ax_semilog.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

1. On which kind of plot is $y = 5 \cdot 3^x$ a straight line? What is its slope (with $\log_{10}$) and where does it cross $x = 0$?
2. A straight line on a log-log plot passes through $(1, 2)$ and $(100, 2000)$. Find the power law $y = Cx^k$.
3. Newton's law of gravity says force $\propto r^{-2}$. What does it look like on a log-log plot? By what factor does the force change when $r$ grows by one decade?

### Intermediate

4. **Estimate it first, then compute.** If transistor counts doubled every two years from $2300$ in 1971, estimate the count in 2021 using $2^{10} \approx 10^3$. Then compute it exactly.
5. Fit Kepler's law to Jupiter's four Galilean moons (distance from Jupiter in km, period in days): Io $(421{,}700;\ 1.769)$, Europa $(671{,}034;\ 3.551)$, Ganymede $(1{,}070{,}412;\ 7.155)$, Callisto $(1{,}882{,}709;\ 16.689)$. What exponent do you find?
6. Generate $y = 3x^{0.75}$ (an exponent like Kleiber's law for metabolic rate) for $x$ from $1$ to $10^4$ with multiplicative noise ($\pm 20\%$), plot it three ways, and fit both an exponential and a power law to it. Which fit is better, and which plot told you so?

### Advanced

7. Show that the slope of a semi-log plot depends on the base of the logarithm but the slope of a log-log plot does not. Convert a semi-log slope of $0.05$ decades per year into a continuous growth rate $r$ and a doubling time.
8. The local slope of a curve on log-log axes is $\dfrac{d \ln y}{d \ln x} = \dfrac{x\, y'}{y}$ (the *elasticity*). Compute it for $y = x^2 + 100$ and show that it moves from $0$ to $2$. Where is it equal to $1$? Explain why this curve is not a power law even though it looks straight for large $x$.
9. Kepler's law in the form $T^2 = \dfrac{4\pi^2}{G M}\, a^3$ says that with $a$ in AU and $T$ in years, $a^3 / T^2$ equals the central mass in units of the Sun's mass. Use the Galilean moons of exercise 5 to estimate Jupiter's mass relative to the Sun ($1\text{ AU} = 1.496 \times 10^{8}$ km, $1$ year $= 365.25$ days).

## 7. Solutions

1. Semi-log ($y$ logarithmic): $\log_{10} y = \log_{10} 5 + x\log_{10} 3$. Slope $\log_{10} 3 \approx 0.477$ decades per unit; it crosses $x = 0$ at $y = 5$.
2. $k = \log_{10}(2000/2) / \log_{10}(100/1) = 3/2 = 1.5$, and $C = 2$ (the value at $x = 1$): $y = 2x^{1.5}$.
3. A straight line of slope $-2$. One decade further away, the force drops by two decades: a factor of $100$.
4. 2021 is $50$ years later: $25$ doublings. $2^{25} = 2^5 \cdot 2^{20} \approx 32 \times 10^6$, so about $2300 \times 3.2 \times 10^{7} \approx 7 \times 10^{10}$. Exactly: $2300 \times 2^{25} = 77{,}175{,}193{,}600 \approx 7.7 \times 10^{10}$. (Real flagship chips around 2021 were in the tens of billions, so the doubling rule held up remarkably well.)
5. The log-log fit gives an exponent of $1.500$ to three decimals — the same law for moons around Jupiter as for planets around the Sun; only the constant differs.
6. The power-law fit recovers $k \approx 0.74$ and $C \approx 3.3$ (close to the true $0.75$ and $3$, given $\pm 20\%$ noise) with residuals of about $0.09$ decades; the exponential fit has residuals about seven times larger, and they are systematically curved. The log-log plot is straight, the semi-log plot bends over.
7. Changing base multiplies every logarithm by the constant $1/\ln b$. On semi-log axes only $y$ is logged, so the slope is scaled by that factor. On log-log axes both rise and run are scaled by the same factor, which cancels. A slope of $0.05$ decades per year means $r = 0.05 \ln 10 \approx 0.1151$ per year (an annual factor $10^{0.05} \approx 1.122$) and a doubling time of $\log_{10} 2 / 0.05 \approx 6.02$ years.
8. $\dfrac{x \cdot 2x}{x^2 + 100} = \dfrac{2x^2}{x^2 + 100}$, which is near $0$ for $x \ll 10$ and tends to $2$ for $x \gg 10$. It equals $1$ when $x^2 = 100$, i.e. $x = 10$. A true power law has **constant** elasticity; this one has slope $\approx 0$ at small $x$ and $\approx 2$ at large $x$, so it only looks like $x^2$ once the constant $100$ is negligible.
9. Converting to AU and years and averaging $a^3/T^2$ over the four moons gives about $9.55 \times 10^{-4}$, i.e. Jupiter is about $1/1047$ of the Sun's mass, in agreement with the accepted value.

Numerical checks:

In [ ]:
import math

import numpy as np

print("Ex 1: slope =", round(math.log10(3), 4), " value at x=0:", 5 * 3**0)
print("Ex 2: k =", math.log10(2000 / 2) / math.log10(100 / 1))
print("Ex 3: force ratio for one decade:", 10.0**-2)
print("Ex 4: estimate", 2300 * 32e6, " exact", 2300 * 2**25)

moon_distance_km = np.array([421_700, 671_034, 1_070_412, 1_882_709])
moon_period_days = np.array([1.769, 3.551, 7.155, 16.689])
k5, c5 = np.polyfit(np.log10(moon_distance_km), np.log10(moon_period_days), 1)
print("Ex 5: exponent =", round(k5, 4))

rng = np.random.default_rng(seed=6)
x6 = np.logspace(0, 4, 40)
y6 = 3 * x6**0.75 * np.exp(rng.normal(0.0, 0.2, size=x6.size))
k6, log_c6 = np.polyfit(np.log10(x6), np.log10(y6), 1)
m6, log_a6 = np.polyfit(x6, np.log10(y6), 1)
power_residual = np.std(np.log10(y6) - (log_c6 + k6 * np.log10(x6)))
exponential_residual = np.std(np.log10(y6) - (log_a6 + m6 * x6))
print(f"Ex 6: power fit k = {k6:.3f}, C = {10**log_c6:.3f}, residual {power_residual:.3f} decades;"
      f" exponential fit residual {exponential_residual:.3f} decades")

print("Ex 7: r =", round(0.05 * math.log(10), 4), " factor", round(10**0.05, 4), " doubling", round(math.log10(2) / 0.05, 2))

x8 = np.array([1.0, 10.0, 100.0, 1000.0])
print("Ex 8: elasticity at x = 1, 10, 100, 1000:", np.round(2 * x8**2 / (x8**2 + 100), 4))

a_au = moon_distance_km / 1.496e8
T_years = moon_period_days / 365.25
mass_ratio = np.mean(a_au**3 / T_years**2)
print(f"Ex 9: M_Jupiter / M_Sun = {mass_ratio:.3e} = 1/{1 / mass_ratio:.0f}")

## 8. Conclusion

- A log axis places $y$ at $\log_{10} y$: equal ratios get equal distances, each decade gets the same room, and data spanning many orders of magnitude becomes readable.
- **Semi-log:** $\;y = A b^x \iff \log_{10} y = \log_{10} A + x\log_{10} b$. Exponentials are straight; the slope $m$ gives the growth factor $10^m$ and the doubling time $\log_{10} 2 / m$.
- **Log-log:** $\;y = C x^k \iff \log_{10} y = \log_{10} C + k\log_{10} x$. Power laws are straight; the slope is the exponent $k$, independent of the log base: $k = $ decades up $/$ decades across.
- Whichever plot straightens the data tells you the model. Fit with `np.polyfit` on logged data, which treats relative errors evenly — right for multiplicative scatter, and essential when values span many decades.
- Kepler's third law $T = a^{1.5}$ is a single straight line through all eight planets, and the same slope holds for Jupiter's moons; the intercept even weighs Jupiter.

**Next:** notebook 04 races the functions against each other — $\ln n$, $n^\varepsilon$, $n$, $n\ln n$, $n^2$, $e^n$ — and uses limits and log-scaled plots to decide who wins and when.